# Toxicity Classification with scikit-learn

In [56]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

import pandas as pd

## Load Jigsaw Toxicity Dataset
Load the [Jigsaw Toxic Comment Classification](https://huggingface.co/datasets/Arsive/toxicity_classification_jigsaw) dataset from Hugging Face. This is the same Wikipedia toxic-comment data from the original Kaggle competition, with labels: `toxic`, `severe_toxic`, `obscene`, `threat`, `insult`, `identity_hate`.

In [4]:
from datasets import load_dataset

# The original google/jigsaw_toxicity_pred uses a deprecated loading script.
# Using a community re-upload in standard parquet/CSV format instead.
ds = load_dataset("Arsive/toxicity_classification_jigsaw")
ds

Generating test split: 100%|██████████| 153164/153164 [00:00<00:00, 200019.36 examples/s]


DatasetDict({
    train: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 25960
    })
    validation: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 6490
    })
    test: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 153164
    })
})

In [62]:
len(ds["test"])/(len(ds["train"])+len(ds["validation"])+len(ds["test"]))

0.8251748251748252

In [93]:
# Preview a few examples
train_df: pd.DataFrame = ds["train"].to_pandas()  # type: ignore[assignment]
train_df = train_df[train_df["toxic"]!=-1]
print(f"Training set size: {len(train_df)}")
train_df.head()

test_df: pd.DataFrame = ds["test"].to_pandas() # type: ignore[assignment]
test_df = test_df[test_df["toxic"]!=-1]
print(test_df.shape)

Training set size: 25960
(63978, 8)


In [94]:
test_df["toxic"].value_counts()

toxic
0    57888
1     6090
Name: count, dtype: int64

In [95]:
train_df["toxic"].value_counts()

toxic
0    13727
1    12233
Name: count, dtype: int64

In [96]:
model = Pipeline([('tfidf', TfidfVectorizer(max_features=5000)),
                  ('lg', LogisticRegression(max_iter=1000))]).fit(train_df["comment_text"],train_df["toxic"])

## Model Training
Train a logistic regression classifier with TF-IDF features.

In [98]:
model.predict_proba(["Monday's suck"])

array([[0.00758443, 0.99241557]])

In [99]:
preds = model.predict(test_df["comment_text"])

In [100]:
print(classification_report(test_df["toxic"],preds))

              precision    recall  f1-score   support

           0       0.98      0.88      0.93     57888
           1       0.42      0.86      0.57      6090

    accuracy                           0.87     63978
   macro avg       0.70      0.87      0.75     63978
weighted avg       0.93      0.87      0.89     63978



## Evaluation

In [101]:
# Detailed breakdown of the classification report
from sklearn.metrics import confusion_matrix

# Confusion matrix
cm = confusion_matrix(test_df["toxic"], preds)
print("Confusion Matrix:")
print(f"True Negatives (TN):  {cm[0,0]:6d}  | False Positives (FP): {cm[0,1]:6d}")
print(f"False Negatives (FN): {cm[1,0]:6d}  | True Positives (TP):  {cm[1,1]:6d}")
print()

# Key metrics explained
print("Metrics for Class 0 (Non-toxic):")
print(f"  Precision: {cm[0,0]/(cm[0,0]+cm[1,0]):.2f}  (of all actual non-toxic, % we correctly called non-toxic)")
print(f"  Recall:    {cm[0,0]/(cm[0,0]+cm[0,1]):.2f}  (of all we called non-toxic, % were actually non-toxic)")
print()

print("Metrics for Class 1 (Toxic):")
print(f"  Precision: {cm[1,1]/(cm[1,1]+cm[0,1]):.2f}  (of all we called toxic, % were actually toxic)")
print(f"  Recall:    {cm[1,1]/(cm[1,1]+cm[1,0]):.2f}  (of all actual toxic, % we correctly called toxic)")

Confusion Matrix:
True Negatives (TN):   50706  | False Positives (FP):   7182
False Negatives (FN):    846  | True Positives (TP):    5244

Metrics for Class 0 (Non-toxic):
  Precision: 0.98  (of all actual non-toxic, % we correctly called non-toxic)
  Recall:    0.88  (of all we called non-toxic, % were actually non-toxic)

Metrics for Class 1 (Toxic):
  Precision: 0.42  (of all we called toxic, % were actually toxic)
  Recall:    0.86  (of all actual toxic, % we correctly called toxic)
